
# Train/Test Splits and Prediction Error

This notebook is intentionally self-contained. It does not import any local utility module, so you can open this file alone during study and see the full statistical workflow, simulation setup, model fitting, evaluation, plotting, and interpretation pattern in one place.



## What problem are we solving?

Train/test evaluation estimates how well a fitted regression method will predict future numeric responses.

**Highest-probability exam pattern:** Compare training error and testing error across model complexities and identify overfitting.



## Assumptions, intuition, and theory

Training MSE is optimistic because the model has already seen those observations. Test MSE is noisier but more honest. Repeating the split or using CV stabilizes the estimate.



    ## Python method notes and documentation

    `train_test_split` creates held-out data, `fit` learns only from the training set, `predict` is applied to the test set, and `mean_squared_error` computes the numeric prediction loss.

    Documentation links:

    - [NumPy random generator](https://numpy.org/doc/stable/reference/random/generator.html)
- [pandas DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)
- [matplotlib pyplot](https://matplotlib.org/stable/api/pyplot_summary.html)
- [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)
- [mean_squared_error](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html)
- [accuracy_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html)
- [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html)
- [make_pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html)
- [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)
- [LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)
- [PolynomialFeatures](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html)


## Fully self-contained worked example

Every code line below is commented so the workflow can be scanned quickly under exam time pressure.

In [ ]:
import numpy as np  # Import numerical arrays and random-number tools.
import pandas as pd  # Import DataFrame tables for simulation summaries.
import matplotlib.pyplot as plt  # Import plotting tools for error curves.
from sklearn.base import clone  # Import clone so repeated simulation fits are independent.
from sklearn.linear_model import LinearRegression  # Import the regression model.
from sklearn.metrics import mean_squared_error  # Import the MSE metric.
from sklearn.model_selection import train_test_split  # Import train/test splitting.
from sklearn.pipeline import make_pipeline  # Import pipeline construction.
from sklearn.preprocessing import PolynomialFeatures  # Import polynomial basis expansion.
RANDOM_SEED = 123  # Store the random seed.
np.random.seed(RANDOM_SEED)  # Fix legacy NumPy randomness.
plt.style.use('default')  # Use a predictable plotting style.


In [ ]:
def make_sine_regression_data(n=120, noise=0.35, random_state=123):  # Define a reusable nonlinear regression simulator.
    rng = np.random.default_rng(random_state)  # Create a reproducible random number generator.
    x = np.sort(rng.uniform(0.0, 3.0, size=n))  # Draw predictor values and sort them for cleaner plots.
    signal = 4.0 + np.sin(3.0 * x)  # Define the true nonlinear mean function.
    y = signal + rng.normal(0.0, noise, size=n)  # Add Gaussian noise to create observed responses.
    X = x.reshape(-1, 1)  # Convert the predictor to a two-dimensional sklearn design matrix.
    return X, y, signal  # Return predictors, observed response, and noiseless truth.

def make_polynomial_regression_data(n=140, noise=2.0, random_state=123):  # Define a curved polynomial regression simulator.
    rng = np.random.default_rng(random_state)  # Create a reproducible random number generator.
    x = rng.uniform(-2.5, 2.5, size=n)  # Draw one numeric predictor over a moderate range.
    signal = 1.0 - 2.0 * x + 0.8 * x**2 + 0.7 * x**3  # Define the true polynomial mean curve.
    y = signal + rng.normal(0.0, noise, size=n)  # Add independent Gaussian errors.
    X = x.reshape(-1, 1)  # Reshape the predictor for sklearn estimators.
    return X, y, signal  # Return the design matrix, response, and true mean.

def train_test_mse(model, X, y, test_size=0.30, random_state=123):  # Define a local train/test regression evaluator.
    X_train, X_test, y_train, y_test = train_test_split(  # Split observations into training and testing parts.
        X,  # Pass the predictor matrix to the splitter.
        y,  # Pass the numeric response vector to the splitter.
        test_size=test_size,  # Hold out this fraction for honest testing.
        random_state=random_state,  # Fix the random split for reproducibility.
    )  # Finish the train/test split call.
    fitted_model = clone(model)  # Clone the estimator so repeated calls do not reuse fitted state.
    fitted_model.fit(X_train, y_train)  # Fit the model only on the training data.
    train_pred = fitted_model.predict(X_train)  # Predict the training responses to assess in-sample error.
    test_pred = fitted_model.predict(X_test)  # Predict the held-out responses to assess future error.
    train_mse = mean_squared_error(y_train, train_pred)  # Compute training mean squared error.
    test_mse = mean_squared_error(y_test, test_pred)  # Compute test mean squared error.
    return fitted_model, train_mse, test_mse  # Return the fitted model and both error estimates.


In [ ]:
rows = []  # Create a list to store replicate-level train/test errors.
for rep in range(40):  # Repeat the full experiment to show split variability.
    X, y, true_signal = make_sine_regression_data(n=130, noise=0.45, random_state=1000 + rep)  # Simulate one data set for this replicate.
    for degree in [1, 3, 8]:  # Compare underfit, reasonable, and flexible polynomial models.
        model = make_pipeline(PolynomialFeatures(degree=degree, include_bias=False), LinearRegression())  # Build the candidate model.
        fitted_model, train_mse, test_mse = train_test_mse(model, X, y, random_state=2000 + rep)  # Estimate train and test error for this replicate.
        rows.append({'rep': rep, 'degree': degree, 'train_mse': train_mse, 'test_mse': test_mse})  # Store the replicate result.
results = pd.DataFrame(rows)  # Convert replicate records to a DataFrame.
summary = results.groupby('degree')[['train_mse', 'test_mse']].mean().reset_index()  # Average errors by model complexity.
display(summary)  # Display the mean error table.
plt.figure(figsize=(6, 4))  # Create an error comparison figure.
plt.plot(summary['degree'], summary['train_mse'], marker='o', label='average train MSE')  # Plot average train error.
plt.plot(summary['degree'], summary['test_mse'], marker='o', label='average test MSE')  # Plot average test error.
plt.xlabel('polynomial degree')  # Label the complexity axis.
plt.ylabel('MSE')  # Label the error axis.
plt.title('Train/test error estimates prediction performance')  # Title the figure.
plt.legend()  # Show the line labels.
plt.show()  # Render the plot.



## Exam-style problem prompt

A prompt asks which of three regression models predicts best. Use a train/test split or repeated train/test simulation, report MSE, and warn if training error and testing error disagree.



    ## Adaptable code pattern

    ```python
    # Step 1: Split the data before fitting.
# Step 2: Fit each candidate model on the training set only.
# Step 3: Predict both training and test responses.
# Step 4: Compare train MSE and test MSE.
# Step 5: Select by test MSE, not by training MSE.
# Step 6: Interpret a large train/test gap as overfitting evidence.
    ```



## What to remember for the exam

The safest exam phrase is: training error estimates how well the model fits known data, while test error estimates future prediction error.
